In [2]:
import pandas as pd

In [4]:
pip install peft==0.9.0

Note: you may need to restart the kernel to use updated packages.


In [5]:
!pip install huggingface_hub[hf_xet]

In [7]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [9]:
!pip install git+https://github.com/tabularis-ai/be_great.git

  Cloning https://github.com/tabularis-ai/be_great.git to c:\users\haava\appdata\local\temp\pip-req-build-4t8qe2q8
  Resolved https://github.com/tabularis-ai/be_great.git to commit 468329e6ef34a5908ad639cf4df1ff654a27799e
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/tabularis-ai/be_great.git 'C:\Users\haava\AppData\Local\Temp\pip-req-build-4t8qe2q8'


In [13]:
df = pd.read_csv(r"C:\Users\haava\Documents\student\student-mat.csv", sep=";")

In [14]:
drop_cols = [
    "G1",
    "G2",
    "guardian",
    "reason",
    "nursery",
    "romantic",
    "schoolsup"
]

df = df.drop(columns=drop_cols)


for c in df.select_dtypes(include="object"):
    df[c] = (
        df[c]
        .astype(str)
        .str.replace(",", " ", regex=False)
        .str.replace("\n", " ", regex=False)
        .str.strip()
    )

df = df.fillna("")

print(df.shape)   # should be ~ (395, 26 or 27)

(395, 26)


In [15]:
categorical = df.select_dtypes(include="object").columns
numeric = df.select_dtypes(exclude="object").columns

df = df[list(categorical) + list(numeric)]

In [16]:
from sklearn.model_selection import train_test_split

D_real_train, D_real_holdout = train_test_split(
    df,
    train_size=300,
    test_size=95,
    random_state=42,
    shuffle=True
)

In [17]:
D_real_train.head().transpose()

,210,75,104,374,16
school,GP,GP,GP,MS,GP
sex,F,M,M,F,F
address,U,U,U,R,U
famsize,GT3,GT3,GT3,LE3,GT3
Pstatus,T,T,A,T,T
Mjob,other,teacher,services,other,services
Fjob,other,other,other,other,services
famsup,yes,yes,yes,no,yes
paid,yes,yes,yes,no,yes
activities,yes,yes,yes,no,yes


In [18]:
import peft
print(peft.__version__)

0.9.0


In [36]:
from be_great import GReaT

model = GReaT(
    llm="facebook/opt-125m",
    batch_size=8,
    epochs=40,
    efficient_finetuning="lora",
    lora_config={
        "r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0,
        "target_modules": ["q_proj", "v_proj"],
    },
    fp16=True,
)

model.fit(D_real_train)

trainable params: 294,912 || all params: 125,534,208 || trainable%: 0.23492560689115113


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
500,2.169100
1000,1.253800
1500,1.182600


In [37]:
#model.tokenizer.pad_token = model.tokenizer.eos_token
#model.model.config.pad_token_id = model.tokenizer.pad_token_id
synthetic = model.sample(
    n_samples=300,
    max_length=200,
    temperature=0.7,
    guided_sampling=True
)

print("Generated:", len(synthetic))
synthetic.to_csv("synthetic_students_opt.csv", index=False)

100%|████████████████████████████████████████████████████████████████████████████████| 300/300 [31:35<00:00,  6.32s/it]

Generated: 300


In [40]:
df_syn_opt = pd.read_csv("synthetic_students_opt.csv", sep=",")

In [42]:
df_syn_opt.shape


(300, 26)

In [44]:
df_syn_opt.head().transpose()

,0,1,2,3,4
school,GP,GP,GP_level,"GP, activities is no",GP
sex,M_1,F,F,F,F
address,U,R,U,U,R
famsize,GT3,GT3,GT3,LE3,LE3
Pstatus,T,T,T,T,T
Mjob,other,services,other,other,services
Fjob,services,other,other,other,services
famsup,no,yes,yes,no,yes
paid,yes,no,no,yes,no
activities,no,no,no,no,no


In [46]:
df.head().transpose()

,0,1,2,3,4
school,GP,GP,GP,GP,GP
sex,F,F,F,F,F
address,U,U,U,U,U
famsize,GT3,GT3,LE3,GT3,GT3
Pstatus,A,T,T,T,T
Mjob,at_home,at_home,at_home,health,other
Fjob,teacher,other,other,services,other
famsup,no,yes,no,yes,yes
paid,no,no,yes,yes,yes
activities,no,no,no,yes,no


In [48]:
def cast_synthetic_to_original_schema(synth: pd.DataFrame, original: pd.DataFrame) -> pd.DataFrame:
    synth = synth.copy()

    synth = synth[[c for c in original.columns if c in synth.columns]]

    for col in synth.columns:
        orig_col = original[col]


        if pd.api.types.is_object_dtype(orig_col) or pd.api.types.is_string_dtype(orig_col):
            synth[col] = synth[col].astype(str)

        elif pd.api.types.is_integer_dtype(orig_col):
            synth[col] = pd.to_numeric(synth[col], errors="coerce")
         
            fill_value = int(orig_col.median())
            synth[col] = synth[col].fillna(fill_value).round().astype(orig_col.dtype)

        elif pd.api.types.is_float_dtype(orig_col):
            synth[col] = pd.to_numeric(synth[col], errors="coerce")

            fill_value = float(orig_col.median())
            synth[col] = synth[col].fillna(fill_value).astype(orig_col.dtype)
 
        elif pd.api.types.is_bool_dtype(orig_col):
            mapping = {
                "true": True, "false": False,
                "1": True, "0": False,
                "yes": True, "no": False
            }
            synth[col] = (
                synth[col]
                .astype(str)
                .str.strip()
                .str.lower()
                .map(mapping)
                .fillna(False)
                .astype(bool)
            )

        else:

            synth[col] = synth[col].astype(orig_col.dtype, errors="ignore")

    return synth

synthetic_casted = cast_synthetic_to_original_schema(df_syn_opt, df)

print(synthetic_casted.dtypes)
synthetic_casted.to_csv("synthetic_students_typed_opt.csv", index=False)

school        object
sex           object
address       object
famsize       object
Pstatus       object
Mjob          object
Fjob          object
famsup        object
paid          object
activities    object
higher        object
internet      object
age            int64
Medu           int64
Fedu           int64
traveltime     int64
studytime      int64
failures       int64
famrel         int64
freetime       int64
goout          int64
Dalc           int64
Walc           int64
health         int64
absences       int64
G3             int64
dtype: object


In [50]:
df_syn_opt_typed = pd.read_csv("synthetic_students_typed_opt.csv", sep=",")

In [52]:
df_syn_opt_typed.shape

(300, 26)

In [54]:
df_syn_opt_typed.head()

,school,sex,address,famsize,Pstatus,Mjob,Fjob,famsup,paid,activities,...,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G3
0,GP,M_1,U,GT3,T,other,services,no,yes,no,...,1,0,4,3,2,2,2,3,0,12
1,GP,F,R,GT3,T,services,other,yes,no,no,...,2,0,4,2,2,2,2,4,3,8
2,GP_level,F,U,GT3,T,other,other,yes,no,no,...,1,0,5,4,5,2,2,4,2,0
3,"GP, activities is no",F,U,LE3,T,other,other,no,yes,no,...,1,0,4,3,1,1,1,1,2,12
4,GP,F,R,LE3,T,services,services,yes,no,no,...,1,0,3,2,4,1,1,3,0,15
